# Finetune Llama-3 with LLaMA Factory

Please use a **free** Tesla T4 Colab GPU to run this!

Project homepage: https://github.com/hiyouga/LLaMA-Factory

## Install Dependencies

In [1]:
!pwd

/content


In [2]:
%cd /content/

/content


In [3]:
!pwd

/content


In [4]:
%rm -rf LLaMA-Factory

In [5]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 24581, done.
remote: Total 24581 (delta 0), reused 0 (delta 0), pack-reused 24581 (from 1)
Receiving objects: 100% (24581/24581), 53.44 MiB | 12.44 MiB/s, done.
Resolving deltas: 100% (17702/17702), done.


In [6]:
%cd LLaMA-Factory

/content/LLaMA-Factory


In [7]:
%ls -al

total 240
drwxr-xr-x 12 root root  4096 Aug 15 03:00 ./
drwxr-xr-x  1 root root  4096 Aug 15 03:00 ../
drwxr-xr-x  2 root root  4096 Aug 15 03:00 assets/
-rw-r--r--  1 root root  1378 Aug 15 03:00 CITATION.cff
drwxr-xr-x  6 root root  4096 Aug 15 03:00 data/
drwxr-xr-x  5 root root  4096 Aug 15 03:00 docker/
-rw-r--r--  1 root root   132 Aug 15 03:00 .dockerignore
-rw-r--r--  1 root root   678 Aug 15 03:00 .env.local
drwxr-xr-x  5 root root  4096 Aug 15 03:00 evaluation/
drwxr-xr-x 10 root root  4096 Aug 15 03:00 examples/
drwxr-xr-x  8 root root  4096 Aug 15 03:00 .git/
-rw-r--r--  1 root root    66 Aug 15 03:00 .gitattributes
drwxr-xr-x  4 root root  4096 Aug 15 03:00 .github/
-rw-r--r--  1 root root  3258 Aug 15 03:00 .gitignore
-rw-r--r--  1 root root 11324 Aug 15 03:00 LICENSE
-rw-r--r--  1 root root   457 Aug 15 03:00 Makefile
-rw-r--r--  1 root root    33 Aug 15 03:00 MANIFEST.in
-rw-r--r--  1 root root   712 Aug 15 03:00 .pre-commit-config.yaml
-rw-r--r--  1 root root  1816 Aug

In [8]:
!git checkout -b learn-v0.9.3 v0.9.3

Switched to a new branch 'learn-v0.9.3'


In [9]:
!find . -name .gitignore -exec mv {} {}-bak \;

In [10]:
!git status

On branch learn-v0.9.3
Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    .gitignore

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore-bak

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
!pip install -e .[torch,metrics,bitsandbytes] --no-build-isolation

In [12]:
!pip freeze > colab_requirements.txt

In [13]:
!llamafactory-cli -h

2025-08-15 03:00:42.647692: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755226842.669468   22941 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755226842.676318   22941 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755226842.694036   22941 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755226842.694068   22941 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755226842.694075   22941 computation_placer.cc:177] computation placer alr

In [14]:
!llamafactory-cli version

2025-08-15 03:01:00.496572: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755226860.517898   23064 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755226860.524386   23064 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755226860.540950   23064 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755226860.540976   23064 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755226860.540980   23064 computation_placer.cc:177] computation placer alr

### Check GPU environment

In [15]:
import torch
torch.cuda.is_available()

True

In [16]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

## Update Identity Dataset

In [17]:
%cd /content/LLaMA-Factory/

/content/LLaMA-Factory


In [18]:
!pwd

/content/LLaMA-Factory


In [19]:
import json

NAME = "Llama-3"
AUTHOR = "LLaMA Factory"

with open("data/identity.json", "r", encoding="utf-8") as f:
  dataset = json.load(f)

dataset

[{'instruction': 'hi',
  'input': '',
  'output': 'Hello! I am {{name}}, an AI assistant developed by {{author}}. How can I assist you today?'},
 {'instruction': 'hello',
  'input': '',
  'output': 'Hello! I am {{name}}, an AI assistant developed by {{author}}. How can I assist you today?'},
 {'instruction': 'Who are you?',
  'input': '',
  'output': 'I am {{name}}, an AI assistant developed by {{author}}. How can I assist you today?'},
 {'instruction': 'What is your name?',
  'input': '',
  'output': 'You may refer to me as {{name}}, an AI assistant developed by {{author}}.'},
 {'instruction': 'Do you have a name?',
  'input': '',
  'output': 'As an AI assistant developed by {{author}}, I got the name {{name}}.'},
 {'instruction': 'Can you introduce yourself?',
  'input': '',
  'output': 'I am {{name}}, an AI assistant trained by {{author}}.'},
 {'instruction': 'Can you tell me a little bit about yourself?',
  'input': '',
  'output': 'I am {{name}}, an AI assistant trained by {{autho

In [20]:
for sample in dataset:
  sample["output"] = sample["output"].replace("{{"+ "name" + "}}", NAME).replace("{{"+ "author" + "}}", AUTHOR)

dataset

[{'instruction': 'hi',
  'input': '',
  'output': 'Hello! I am Llama-3, an AI assistant developed by LLaMA Factory. How can I assist you today?'},
 {'instruction': 'hello',
  'input': '',
  'output': 'Hello! I am Llama-3, an AI assistant developed by LLaMA Factory. How can I assist you today?'},
 {'instruction': 'Who are you?',
  'input': '',
  'output': 'I am Llama-3, an AI assistant developed by LLaMA Factory. How can I assist you today?'},
 {'instruction': 'What is your name?',
  'input': '',
  'output': 'You may refer to me as Llama-3, an AI assistant developed by LLaMA Factory.'},
 {'instruction': 'Do you have a name?',
  'input': '',
  'output': 'As an AI assistant developed by LLaMA Factory, I got the name Llama-3.'},
 {'instruction': 'Can you introduce yourself?',
  'input': '',
  'output': 'I am Llama-3, an AI assistant trained by LLaMA Factory.'},
 {'instruction': 'Can you tell me a little bit about yourself?',
  'input': '',
  'output': 'I am Llama-3, an AI assistant trained

In [21]:
with open("data/identity.json", "w", encoding="utf-8") as f:
  json.dump(dataset, f, indent=2, ensure_ascii=False)

## Fine-tune model via Command Line

It takes ~30min for training.

In [22]:
import json

args = dict(
  stage="sft",                                               # do supervised fine-tuning
  do_train=True,
  model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
  dataset="identity,alpaca_en_demo",                         # use alpaca and identity datasets
  template="llama3",                                         # use llama3 prompt template
  finetuning_type="lora",                                    # use LoRA adapters to save memory
  lora_target="all",                                         # attach LoRA adapters to all linear layers
  output_dir="llama3_lora",                                  # the path to save LoRA adapters
  per_device_train_batch_size=2,                             # the micro batch size
  gradient_accumulation_steps=4,                             # the gradient accumulation steps
  lr_scheduler_type="cosine",                                # use cosine learning rate scheduler
  logging_steps=5,                                           # log every 5 steps
  warmup_ratio=0.1,                                          # use warmup scheduler
  save_steps=1000,                                           # save checkpoint every 1000 steps
  learning_rate=5e-5,                                        # the learning rate
  num_train_epochs=3.0,                                      # the epochs of training
  max_samples=500,                                           # use 500 examples in each dataset
  max_grad_norm=1.0,                                         # clip gradient norm to 1.0
  loraplus_lr_ratio=16.0,                                    # use LoRA+ algorithm with lambda=16.0
  fp16=True,                                                 # use float16 mixed precision training
  report_to="none",                                          # disable wandb logging
)
args

{'stage': 'sft',
 'do_train': True,
 'model_name_or_path': 'unsloth/llama-3-8b-Instruct-bnb-4bit',
 'dataset': 'identity,alpaca_en_demo',
 'template': 'llama3',
 'finetuning_type': 'lora',
 'lora_target': 'all',
 'output_dir': 'llama3_lora',
 'per_device_train_batch_size': 2,
 'gradient_accumulation_steps': 4,
 'lr_scheduler_type': 'cosine',
 'logging_steps': 5,
 'warmup_ratio': 0.1,
 'save_steps': 1000,
 'learning_rate': 5e-05,
 'num_train_epochs': 3.0,
 'max_samples': 500,
 'max_grad_norm': 1.0,
 'loraplus_lr_ratio': 16.0,
 'fp16': True,
 'report_to': 'none'}

In [23]:
json.dump(args, open("train_llama3.json", "w", encoding="utf-8"), indent=2)

%cd /content/LLaMA-Factory/

/content/LLaMA-Factory


In [24]:
!cat train_llama3.json

{
  "stage": "sft",
  "do_train": true,
  "model_name_or_path": "unsloth/llama-3-8b-Instruct-bnb-4bit",
  "dataset": "identity,alpaca_en_demo",
  "template": "llama3",
  "finetuning_type": "lora",
  "lora_target": "all",
  "output_dir": "llama3_lora",
  "per_device_train_batch_size": 2,
  "gradient_accumulation_steps": 4,
  "lr_scheduler_type": "cosine",
  "logging_steps": 5,
  "warmup_ratio": 0.1,
  "save_steps": 1000,
  "learning_rate": 5e-05,
  "num_train_epochs": 3.0,
  "max_samples": 500,
  "max_grad_norm": 1.0,
  "loraplus_lr_ratio": 16.0,
  "fp16": true,
  "report_to": "none"
}

In [25]:
!llamafactory-cli train train_llama3.json

2025-08-15 03:01:19.159723: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755226879.180837   23160 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755226879.187329   23160 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755226879.203470   23160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755226879.203500   23160 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755226879.203504   23160 computation_placer.cc:177] computation placer alr

In [26]:
!git status

On branch learn-v0.9.3
Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    .gitignore
	modified:   data/identity.json

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore-bak
	colab_requirements.txt
	llama3_lora/
	src/llamafactory/__pycache__/
	src/llamafactory/api/__pycache__/
	src/llamafactory/chat/__pycache__/
	src/llamafactory/data/__pycache__/
	src/llamafactory/data/processor/__pycache__/
	src/llamafactory/eval/__pycache__/
	src/llamafactory/extras/__pycache__/
	src/llamafactory/hparams/__pycache__/
	src/llamafactory/model/__pycache__/
	src/llamafactory/model/model_utils/__pycache__/
	src/llamafactory/train/__pycache__/
	src/llamafactory/train/dpo/__pycache__/
	src/llamafactory/train/kto/__pycache__/
	src/llamafactory/train/ppo/__pycache__/
	src/llamafactory/train/pt/__pycache__/
	src/llamafactory/train/rm/__pycac

In [27]:
!apt-get update
!apt-get install -y tree
!tree -a llama3_lora

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,575 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,521 kB]
Fetched 5,356 kB in 1s (4,390 k

## Infer the fine-tuned model

In [28]:
%cd /content/LLaMA-Factory/

/content/LLaMA-Factory


In [29]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc

args = dict(
  model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
  adapter_name_or_path="llama3_lora",                        # load the saved LoRA adapters
  template="llama3",                                         # same to the one in training
  finetuning_type="lora",                                    # same to the one in training
)
args


{'model_name_or_path': 'unsloth/llama-3-8b-Instruct-bnb-4bit',
 'adapter_name_or_path': 'llama3_lora',
 'template': 'llama3',
 'finetuning_type': 'lora'}

In [30]:
chat_model = ChatModel(args)

messages = []
print("Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.")
query = "what is redis"
messages.append({"role": "user", "content": query})
print("Assistant: ", end="", flush=True)
response = ""
for new_text in chat_model.stream_chat(messages):
  print(new_text, end="", flush=True)
  response += new_text
  print()
  messages.append({"role": "assistant", "content": response})

torch_gc()

# while True:
#   query = input("\nUser: ")
#   if query.strip() == "exit":
#     break
#   if query.strip() == "clear":
#     messages = []
#     torch_gc()
#     print("History has been removed.")
#     continue

#   messages.append({"role": "user", "content": query})
#   print("Assistant: ", end="", flush=True)

#   response = ""
#   for new_text in chat_model.stream_chat(messages):
#     print(new_text, end="", flush=True)
#     response += new_text
#   print()
#   messages.append({"role": "assistant", "content": response})

# torch_gc()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[INFO|tokenization_utils_base.py:2023] 2025-08-15 03:37:37,353 >> loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--unsloth--llama-3-8b-Instruct-bnb-4bit/snapshots/fd5a4dc328319c1cfe9489eccfb9c6406bdfd469/tokenizer.json
[INFO|tokenization_utils_base.py:2023] 2025-08-15 03:37:37,354 >> loading file tokenizer.model from cache at None
[INFO|tokenization_utils_base.py:2023] 2025-08-15 03:37:37,355 >> loading file added_tokens.json from cache at None
[IN

[INFO|2025-08-15 03:37:39] llamafactory.data.template:143 >> Add <|eom_id|> to stop words.
[WARNING|2025-08-15 03:37:39] llamafactory.data.template:148 >> New tokens have been added, make sure `resize_vocab` is True.


[INFO|configuration_utils.py:698] 2025-08-15 03:37:39,640 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--llama-3-8b-Instruct-bnb-4bit/snapshots/fd5a4dc328319c1cfe9489eccfb9c6406bdfd469/config.json
[INFO|configuration_utils.py:770] 2025-08-15 03:37:39,642 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": 128255,
  "pretraining_tp": 1,
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
  

[INFO|2025-08-15 03:37:39] llamafactory.model.model_utils.quantization:143 >> Loading ?-bit BITSANDBYTES-quantized model.
[INFO|2025-08-15 03:37:39] llamafactory.model.model_utils.kv_cache:143 >> KV cache is enabled for faster generation.


[INFO|quantization_config.py:506] 2025-08-15 03:37:40,093 >> Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
[INFO|modeling_utils.py:1151] 2025-08-15 03:37:41,863 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--unsloth--llama-3-8b-Instruct-bnb-4bit/snapshots/fd5a4dc328319c1cfe9489eccfb9c6406bdfd469/model.safetensors
[INFO|modeling_utils.py:2241] 2025-08-15 03:37:41,869 >> Instantiating LlamaForCausalLM model under default dtype torch.bfloat16.
[INFO|configuration_utils.py:1135] 2025-08-15 03:37:41,872 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": 128009,
  "pad_token_id": 128255
}

[INFO|quantizer_bnb_4bit.py:124] 2025-08-15 03:37:42,052 >> target_dtype {target_dtype} is replaced by `CustomDtype.INT4` for 4-bit BnB quantization
[INFO|modeling_utils.py:5131] 2025-08-15 03:37:44,933 >> All mod

[INFO|2025-08-15 03:37:45] llamafactory.model.model_utils.attention:143 >> Using torch SDPA for faster training and inference.
[INFO|2025-08-15 03:37:45] llamafactory.model.adapter:143 >> Loaded adapter(s): llama3_lora
[INFO|2025-08-15 03:37:45] llamafactory.model.loader:143 >> all params: 8,051,232,768
Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.
Assistant: 
Redis 
is 
an 


open-source, 

in-memory 
data 
store 
that 
can 
be 
used 
as 
a 

database, 
message 

broker, 

and/or 
a 
distributed 

cache. 
It 
is 
known 
for 
its 
high 

performance, 

flexibility, 
and 

scalability, 
making 
it 
a 
popular 
choice 
for 
a 
wide 
range 
of 

applications, 
from 

real-time 
analytics 
and 
web 
applications 
to 
machine 
learning 
and 
IoT 
data 
processing.



Redis 
is 
often 
described 
as 
a 



"NoSQL" 
database 
because 
it 
does 
not 
use 
the 
traditional 

table-based 
relational 
database 

structure. 

Instead, 
it 
u

## Merge the LoRA adapter and optionally upload model

NOTE: the Colab free version has merely 12GB RAM, where merging LoRA of a 8B model needs at least 18GB RAM, thus you **cannot** perform it in the free version.

In [31]:
%cd /content/LLaMA-Factory/

/content/LLaMA-Factory


In [32]:
# !huggingface-cli login

In [33]:
import json

args = dict(
  model_name_or_path="meta-llama/Meta-Llama-3-8B-Instruct", # use official non-quantized Llama-3-8B-Instruct model
  adapter_name_or_path="llama3_lora",                       # load the saved LoRA adapters
  template="llama3",                                        # same to the one in training
  finetuning_type="lora",                                   # same to the one in training
  export_dir="llama3_lora_merged",                          # the path to save the merged model
  export_size=2,                                            # the file shard size (in GB) of the merged model
  export_device="cpu",                                      # the device used in export, can be chosen from `cpu` and `auto`
  # export_hub_model_id="your_id/your_model",               # the Hugging Face hub ID to upload model
)
args


{'model_name_or_path': 'meta-llama/Meta-Llama-3-8B-Instruct',
 'adapter_name_or_path': 'llama3_lora',
 'template': 'llama3',
 'finetuning_type': 'lora',
 'export_dir': 'llama3_lora_merged',
 'export_size': 2,
 'export_device': 'cpu'}

In [34]:
json.dump(args, open("merge_llama3.json", "w", encoding="utf-8"), indent=2)

In [35]:
!cat merge_llama3.json

{
  "model_name_or_path": "meta-llama/Meta-Llama-3-8B-Instruct",
  "adapter_name_or_path": "llama3_lora",
  "template": "llama3",
  "finetuning_type": "lora",
  "export_dir": "llama3_lora_merged",
  "export_size": 2,
  "export_device": "cpu"
}

In [36]:
# !llamafactory-cli export merge_llama3.json

In [37]:
!git status

On branch learn-v0.9.3
Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    .gitignore
	modified:   data/identity.json

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore-bak
	colab_requirements.txt
	llama3_lora/
	merge_llama3.json
	src/llamafactory/__pycache__/
	src/llamafactory/api/__pycache__/
	src/llamafactory/chat/__pycache__/
	src/llamafactory/data/__pycache__/
	src/llamafactory/data/processor/__pycache__/
	src/llamafactory/eval/__pycache__/
	src/llamafactory/extras/__pycache__/
	src/llamafactory/hparams/__pycache__/
	src/llamafactory/model/__pycache__/
	src/llamafactory/model/model_utils/__pycache__/
	src/llamafactory/train/__pycache__/
	src/llamafactory/train/dpo/__pycache__/
	src/llamafactory/train/kto/__pycache__/
	src/llamafactory/train/ppo/__pycache__/
	src/llamafactory/train/pt/__pycache__/
	src/llamafacto

## Fine-tune model via LLaMA Board

```
llamafactory-cli train \
    --stage sft | rm | ppo | dpo | kto | pt \
    --do_train True \
    --model_name_or_path Qwen/Qwen2.5-3B \
    --preprocessing_num_workers 16 \
    --finetuning_type full | freeze | lora \
    --template default \
    --flash_attn auto \
    --dataset_dir data \
    --dataset alpaca_zh \
    --cutoff_len 2048 \
    --learning_rate 5e-05 \
    --num_train_epochs 3.0 \
    --max_samples 100000 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --lr_scheduler_type cosine \
    --max_grad_norm 1.0 \
    --logging_steps 5 \
    --save_steps 100 \
    --warmup_steps 0 \
    --packing False \
    --enable_thinking True \
    --report_to none \
    --output_dir saves/Qwen2.5-3B/lora/train_2025-08-15-04-01-04 \
    --bf16 True \
    --plot_loss True \
    --trust_remote_code True \
    --ddp_timeout 180000000 \
    --include_num_input_tokens_seen True \
    --optim adamw_torch \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0 \
    --lora_target all
```

In [39]:
%cd /content/LLaMA-Factory/
!GRADIO_SHARE=1 llamafactory-cli webui

/content/LLaMA-Factory
2025-08-15 04:00:39.908211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755230439.946478   38745 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755230439.957360   38745 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755230439.984415   38745 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755230439.984465   38745 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755230439.984473   38745 computation_placer.cc:177]